# 05_regression_analysis.ipynb

## **Objective:**
Build, evaluate, and interpret regression models to predict customer spend based on engineered features.

---

## **1. Import Necessary Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

## **2. Set Up Logging**

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## **3. Define File Paths**

In [ ]:
ENGINEERED_DATA_PATH = "../data/engineered_data.csv"
REGRESSION_DATA_PATH = "../data/regression_data.csv"

# Make sure the data directory exists
os.makedirs(os.path.dirname(REGRESSION_DATA_PATH), exist_ok=True)

## **4. Load Dataset**

In [ ]:
def load_data(file_path):
    """Load dataset safely."""
    if not os.path.exists(file_path):
        logging.error(f"File not found: {file_path}")
        return None
    try:
        df = pd.read_csv(file_path)
        logging.info("Data loaded successfully.")
        return df
    except Exception as e:
        logging.error(f"Error loading data: {e}")
        return None
    
df = load_data(ENGINEERED_DATA_PATH)

## **5. Inspect and Select Features**

In [ ]:
target = 'Total Spend'
drop_cols = ['Customer ID', 'Purchase Date', 'Gender', 'Payment Method', 'Shipping Type', 'Product Type', 'Add-ons Purchased', 'Loyalty Member', 'SKU', 'Order Status', 'Purchase Weekday']
X = df.drop(columns=[target] + drop_cols, errors='ignore')
y = df[target]

logging.info(f"Features shape: {X.shape}, Target shape: {y.shape}")

## **6. Train/Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logging.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## **7. Feature scaling**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logging.info("Data scaling completed.")

## **8. Train Basic Models**

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest' : RandomForestRegressor(random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(random_state=42)
}

results_dict = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results_dict[name] = {
        'MAE' : mae,
        'RMSE' : rmse,
        'R2' : r2,
        'Predictions' : y_pred,
        'Model' : model
    }
    logging.info(f"{name} - MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}")

## **9. GridSearch for Random Forest**

In [ ]:
print("\nTuning Random FOrest with GridSearchCV...")
rf_params = {
    'n_estimators' : [100, 200],
    'max_depth' : [None, 10, 20],
    'min_samples_split' : [2, 5]
}
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
rf_grid.fit(X_train_scaled, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test_scaled)

results_dict['Tuned Random Forest'] = {
    'MAE' : mean_absolute_error(y_test, y_pred_rf),
    'RMSE' : np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'R2' : r2_score(y_test, y_pred_rf),
    'Predictions' : y_pred_rf,
    'Model' : best_rf
}
logging.info(f"Tuned Random Forest - MAE: {results_dict['Tuned Random Forest']['MAE']:.2f}, RMSE: {results_dict['Tuned Random Forest']['RMSE']:.2f}, R2: {results_dict['Tuned Random Forest']['R2']:.2f}")
logging.info("Random Forest predictions completed.")

## **10. GridSearch for Gradient Boosting**

In [ ]:
print("\nTuning Gradient Boosting with GradientSearchCV...")
gb_params = {
    'n_estimators' : [100, 150],
    'learning_rate' : [0.05, 0.1],
    'max_depth' : [3, 5]
}
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
gb_grid.fit(X_train_scaled, y_train)
best_gb = gb_grid.best_estimator_
y_pred_gb = best_gb.predict(X_test_scaled)

results_dict['Tuned Gradient Boosting'] = {
    'MAE' : mean_absolute_error(y_test, y_pred_gb),
    'RMSE' : np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    'R2' : r2_score(y_test, y_pred_gb),
    'Predictions' : y_pred_gb,
    'Model' : best_gb
}
logging.info(f"Tuned Gradient Boosting - MAE: {results_dict['Tuned Gradient Boosting']['MAE']:.2f}, RMSE: {results_dict['Tuned Gradient Boosting']['RMSE']:.2f}, R2: {results_dict['Tuned Gradient Boosting']['R2']:.2f}")
logging.info("Gradient Boosting predictions completed.")

## **11. Compare Model Performances**

In [ ]:
comparison = pd.DataFrame(results_dict).T[['MAE', 'RMSE', 'R2']]
print("\nModel Comparison:")
print(comparison)

logging.info("Model comparison completed.")

## **12. Feature Importance**

In [ ]:
feature_names = X.columns

def plot_feature_importance(model, model_name):
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        indices = np.argsort(importance)[::-1]
        plt.figure(figsize=(10,6))
        plt.title(f'{model_name} - Feature Importance')
        sns.barplot(x=importance[indices], y=feature_names[indices])
        plt.tight_layout()
        plt.show()

plot_feature_importance(best_rf, 'Tuned Random Forest')
plot_feature_importance(best_gb, 'Tuned Gradient Boosting')

logging.info("Feature importance plots completed.")

## **13. Save Best Model Results**

In [ ]:
def save_model(file_path):    
    best_model_name = max(results_dict.items(), key=lambda x: x[1]['R2'])[0]
    best_predictions = results_dict[best_model_name]['Predictions']

    final_results = pd.DataFrame({
        'Actual' : y_test,
        'Predicted' : best_predictions
    })
    final_results.to_csv(file_path, index=False)
    logging.info(f"Final results saved to {file_path}")

save_model(REGRESSION_DATA_PATH)

## **Summary & Next Steps**
✅ Regression model trained and evaluated.  
✅ Results saved for further analysis.  
➡️ Next, proceed to `06_visualization_export.ipynb` to visualize insights.